# E01 Iris：小样本分层验证与标签提交

> Issue #9 / Epic #6 / 路线图 E01

## 你将学到
1. 小样本多分类为什么必须分层划分
2. 整数标签 vs 字符串标签如何正确生成 submission
3. 用 `wanglaoshi` 完成：环境检查 → EDA → 基线 → 提交校验

## 技术难点
- 每类只有 50 条，验证集波动大
- `target` 数字标签与 `species` 字符串标签提交格式不同
- 禁止对概率矩阵静默 argmax（见 B05 修复）


In [ ]:
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE

JE.no_warning()
V.show_env_info()


## 1. 加载 Iris，同时保留数字标签与字符串标签


In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species'] = iris.target_names[iris.target]
print('shape =', df.shape)
print('target counts:\n', df['target'].value_counts().sort_index())
print('species counts:\n', df['species'].value_counts())
df.head()


## 2. EDA：小样本下先看分布与相关


In [ ]:
from wanglaoshi import DataAnalyzer

analyzer = DataAnalyzer(df)
print(analyzer.get_basic_info())
display(analyzer.basic_statistics())
display(analyzer.missing_value_analysis())


## 3. 分层划分：对比非分层的风险


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in df.columns if c not in ('target', 'species')]
X = df[feature_cols]
y_int = df['target']
y_str = df['species']

# 分层划分（正确）
Xtr, Xte, ytr, yte = train_test_split(
    X, y_int, test_size=0.3, random_state=RANDOM_STATE, stratify=y_int
)
print('stratified train:', ytr.value_counts().sort_index().to_dict())
print('stratified test :', yte.value_counts().sort_index().to_dict())

# 非分层（错误示范：某折可能缺类）
Xtr_bad, Xte_bad, ytr_bad, yte_bad = train_test_split(
    X, y_int, test_size=0.3, random_state=7, stratify=None
)
print('non-strat train:', ytr_bad.value_counts().sort_index().to_dict())

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=5 if not QUICK else 3, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(pipe, X, y_int, cv=cv, scoring='accuracy')
print(f'CV accuracy mean±std = {scores.mean():.3f} ± {scores.std():.3f}')


## 4. QuickBaseline 对比


In [ ]:
from wanglaoshi import CompetitionTools as CT

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr, ytr, Xte, yte, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


## 5. 提交：整数标签 vs 字符串标签


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

best = comparison.iloc[0]['Model']
pred_int = results[best]['predictions']
out = Path('outputs/01_iris'); out.mkdir(parents=True, exist_ok=True)

# A) 整数标签提交
sample_int = pd.DataFrame({'Id': np.arange(len(pred_int)), 'target': yte.to_numpy()})
sample_int.to_csv(out / 'sample_int.csv', index=False)
gen = CT.SubmissionGenerator()
sub_int = gen.create_submission(
    pred_int, str(out / 'sample_int.csv'), str(out / 'sub_int.csv'), prediction_type='label'
)
print('INT', gen.validate_submission(str(out / 'sub_int.csv'), str(out / 'sample_int.csv')))

# B) 字符串标签提交（把数字预测映射回物种名）
pred_str = iris.target_names[np.asarray(pred_int).astype(int)]
sample_str = pd.DataFrame({'Id': np.arange(len(pred_str)), 'species': y_str.loc[yte.index].to_numpy()})
sample_str.to_csv(out / 'sample_str.csv', index=False)
sub_str = gen.create_submission(
    pred_str, str(out / 'sample_str.csv'), str(out / 'sub_str.csv'),
    prediction_type='label', target_column='species'
)
print('STR', gen.validate_submission(str(out / 'sub_str.csv'), str(out / 'sample_str.csv')))
display(sub_int.head(3)); display(sub_str.head(3))


## 6. 误差分析


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(yte, pred_int, target_names=list(iris.target_names)))
print(confusion_matrix(yte, pred_int))


## 7. 检查清单
- [x] 分层划分与 CV 方差
- [x] 整数/字符串两种提交
- [x] SubmissionGenerator 校验
- [ ] 用概率提交时显式指定 prediction_type='probability'（禁止静默 argmax）
